### MY470 Computer Programming

### Problem Set 2, AT 2025

#### \*\*\* Due 12:00 noon on Monday, October 27 \*\*\*

---
### Writing your own k-means clustering algorithm

K-means clustering is a simple unsupervised machine-learning method for cluster analysis. The aim of the method is to partition a set of points into k clusters, such that each point is assigned to the nearest cluster. The algorithm iterates through two steps:

1. Assign each data point to the cluster with the nearest centroid
2. Update the centroids of the clusters given the new assignment

The algorithm converges when the assignments no longer change. Since the intial assignment to clusters is largely random, there is no guarantee that the optimum assignment is found. So it is common to run the algorithm multiple times and use different starting conditions.

In this problem set, we will implement a much simplified version of the k-means clustering algorithm. Rather than running the algorithm until convergence, we will repeat the above two steps a large but fixed number of times. In addition, we will initialize only once, using a naive method according to which we randomly choose k points from the data to use as initial cluster centroids. 

(In real life, you will of course use a library to implement such an algorithm. In Python, you can do this using [scikit-learn](http://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).)

For the problem set, we will additionally use data from the file `Wholesale customers data.csv`, which you can find in the `data` repository. The file contains information on the annual spending on diverse product categories for the clients of a wholesale distributor. The data are obtained from the [UCI Machine Learning Repository](http://archive.ics.uci.edu/ml/index.php) and you can find more information about them [here](http://archive.ics.uci.edu/ml/datasets/Wholesale+customers#).

#### Please note!

Always follow our specifications exactly, or you will lose points. 

Use docstrings to describe your functions. We will subtract points from your mark if you do not use appropriate description of your code.

There are many different implementations of the k-means algorithm you can find online. However, this problem set expects you to follow the instructions and algorithms below precisely.  

In [1]:
# We will first import the modules we need. (Imports should always 
# go on top of all code to signal if any installations are needed).
# You are expected to solve the problem set with these modules only.
# Do not import and use any other ones.

# You will need the math module to estimate the square root.
# To get the square root of num, use math.sqrt(num)
import math
import csv
import random

### Problem 1: Function to estimate Euclidean distance between two points

The function should take two lists as arguments, where each list contains the n coordinates of each of the two points. 

Write a function that calculates the Euclidean distance between two n-dimensional points. The function should follow these specifications:
* It is called `get_distance`.
* It takes two lists as parameters. Each list contains the coordinates of a point in n-dimensional Euclidean space.
* It returns the numeric distance (float) between the two points.

#### Hints

You can read about the definition of Euclidean distance on [Wikipedia](https://en.wikipedia.org/wiki/Euclidean_distance).


In [2]:
def get_distance(x, y):
    """Calculates the Euclidean distance between
    a pair of n-dimensional coordinates. It must
    have 2 or more dimensions.
    """
    # Calculates Euclidean distance
    ed = sum([((x[i] - y[i])**2) for i in range(len(x))]) ** 0.5
    
    return ed

### Test case for Problem 1

Test your `get_distance` function for the points [0, 3, 0] and [4, 0, 0].

In [3]:
x = [0, 3, 0]
y = [4, 0, 0]
get_distance(x, y)

5.0

### Problem 2: Function to estimate the centroid of a collection of points

Write a function that estimates the centroid of a collection of n-dimensional points. The specifications are as follows: 
* The function is called `get_centroid`.
* It takes one list as a parameter, which contains each of the points entered as a list of n coordinates. 
* The function returns a list with the coordinates of the virtual center point.

#### Hints

The coordinate of the centroid in each dimension is the mean of the coordinates of all the points in that dimension.

In [4]:
def get_centroid(ls):
    """Given a collection of n-dimensional points,
    calculates the coordinates of the centroid.
    It takes a list of lists as a parameter.
    Each list contains the coordinates of a point
    in n-dimensional Euclidean space.
    """
    dimensions = range(len(ls[0]))
    
    # Get pair-values (dimension-value)
    pair_values = [[a, b[a]] for a in dimensions for b in ls]
    
    # Assign to each dimension a set of values
    dic = {d: [] for d in dimensions}
    
    for i, j in pair_values:
        dic[i].append(j)
    
    # Calculate centroid
    centroid = [sum(dic[key])/len(dic[key]) for key in dic.keys()]
    return centroid

### Test case for Problem 2

Test `get_centroid` below using `test_lst`. Print the returned centroid.

In [5]:
test_lst = [[0,0,0], [0,0,1], [0,1,0], [1,0,0], 
            [0,1,1], [1,0,1], [1,1,0], [1,1,1]]

get_centroid(test_lst)

[0.5, 0.5, 0.5]

---
### Problem 3: Function to read data

Write a function that opens a csv file similar to `../data/Wholesale customers data.csv` and returns part of its data. The specifications are:
* The function is called `get_data`.
* It takes a string as a parameter, which is the location of the file.
* It returns the data as a list. Please use the csv module to read the file. You can read how to do this [here](https://docs.python.org/3/library/csv.html). Make sure you do not include the column names in the data. 
* Each element in the list you return corresponds to a row of the file. Each element is a list of the values for that row for column 3, 4, 5, etc. (skip the values for columns 1 and 2). Make sure that the values are of appropriate data types. In the case of the example data, your list should contain 440 elements (customers), each of which is a list of six numeric values (amounts spent on products). 


In [6]:
def get_data(path, sep = ',', skip_cols = 0):
    """Read a CSV file and returns the data without headers as a list.
    You can skip the number of columns you want.
    """
    
    with open(path, newline='') as csvfile:
        # Read CSV file
        read_file = csv.reader(csvfile, delimiter = sep, quotechar='|')
        
        # Skip headers
        next(read_file, None)
            
        # Get list of lists with values skipping columns if needed
        lst = [row[skip_cols:] for row in read_file]
        
        # Convert values to numbers
        lst = [[int(j) for j in i] for i in lst]

        return lst

### Test case for Problem 3

Test your function below.
* Use `../data/Wholesale customers data.csv` as an argument for the function.
* Save the data it returns in a variable called `data`. 
* Then print the first two elements of `data`.

In [7]:
# Read CSV file
data = get_data("../data/Wholesale customers data.csv", skip_cols = 2)

# Print firs two elements
print(data[:2])

[[12669, 9656, 7561, 214, 2674, 1338], [7057, 9810, 9568, 1762, 3293, 1776]]


---
### Problem 4: Function to implement k-means algorithm

Write a function called `kmeans` that clusters a collection of points into k clusters using a simplified version of the k-means algorithm. The function has two parameters: 

1. `points` – a list of n-dimensional points;
2. `k` – an integer that defines the number of desired clusters.

The function should return a tuple of two elements in the following sequence: 

1. A list of `k` clusters, each of which is a list of points (each of which is a list of coordinates);
2. A list of the centroids for each of the `k` clusters. Each centroid is essentially a point, so it should be presented as a list of coordinates.

Write your code around the detailed comments and the helping code below. Do not set any random seeds inside this function.

In [8]:
def kmeans(points, k, iter = 100):
    """Takes a list of lists with n-dimensional points
    and clusters it into k number of clusters.
    """
    
    # Select k random points to use as initial centroids
    init = random.sample(points, k)
    
    # Create a list to keep the centroids of the k clusters
    # For now, this list will contain the points from init
    centroids = [i for i in init]
    
    # Iterate n times the process so the centroids will become stable
    for iteration in range(iter):
        # Create a list of k lists to contain the points assigned to each cluster  
        clusters = [[] for i in init]
        
        # Assign each point to the cluster with the closest centroid
        for i in points:
            # Get initial distance for comparison
            min_index = 0
            min_dist = get_distance(i, centroids[0])
            
            # Compare distance to each centroid
            for j in range(1, len(centroids)):
                dist = get_distance(i, centroids[j])
                if dist < min_dist:
                    min_dist = dist
                    min_index = j
            
            # Assign to the closest centroid
            clusters[min_index].append(i)
        
        # Estimate the new centroids
        centroids = []
        
        for c in clusters:
            centroids.append(get_centroid(c))
    
    return (clusters, centroids)

### Test case for Problem 4
Test your function on the data from Problem 3 for k = 3. For each of the three clusters:
* print the number of customers assigned to it, and
* print the cordinates of its centroid.

In [9]:
k = 3
results = kmeans(data, k)

for i in range(k):
    print(f"Cluster {i + 1} has {len(results[0][i])} customers assigned. \nThe coordinates of its centroid are:")
    print(results[1][i])
    print("")

Cluster 1 has 53 customers assigned. 
The coordinates of its centroid are:
[7751.981132075472, 17910.509433962263, 27037.905660377357, 1970.9433962264152, 12104.867924528302, 2185.735849056604]

Cluster 2 has 328 customers assigned. 
The coordinates of its centroid are:
[8341.612804878048, 3779.893292682927, 5152.173780487805, 2577.237804878049, 1720.5731707317073, 1136.5426829268292]

Cluster 3 has 59 customers assigned. 
The coordinates of its centroid are:
[36156.38983050847, 6123.64406779661, 6366.779661016949, 6811.118644067797, 1050.0169491525423, 3090.0508474576272]



---

### Evaluation

| Problem | Mark     | Comment   
|:-------:|:--------:|:----------------------
| 1       |   /2    |              
| 2       |   /2    | 
| 3       |   /2    | 
| 4       |   /6    | 
| Legibility      |   /2    | 
| Modularity      |   /2    | 
| Efficiency      |   /4    | 
|**Total**|**/20**  | 